# 02 - Model Building & Evaluation

This notebook covers:
- Loading the cleaned dataset
- Train-test split and scaling
- Training multiple classifiers
- Cross-validation and hyperparameter tuning
- Model comparison
- Final model selection and saving

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve)

import xgboost as xgb
import lightgbm as lgb
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries imported successfully')

### Load Cleaned Data

In [ ]:
df = pd.read_csv('../Data/diabetes_cleaned.csv')
print('Shape:', df.shape)
df.head()

### Train-Test Split

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'Train target distribution:\n{y_train.value_counts(normalize=True)}')

### Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler for the app
joblib.dump(scaler, '../scaler.pkl')
print('Scaler saved as scaler.pkl')

### Baseline Model Training

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1)
}

results = []
trained_models = {}

for name, model in models.items():
    if name in ['Logistic Regression', 'SVM']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    trained_models[name] = model
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })
    
    print(f'{name} trained.')

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df

### Results Summary

In [ ]:
print('\n=== Model Comparison ===\n')
print(results_df.to_string(index=False))

# Plot comparison
results_df.set_index('Model').plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.legend(loc='lower right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    if name in ['Logistic Regression', 'SVM']:
        scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    else:
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    
    cv_results.append({
        'Model': name,
        'CV Mean ROC-AUC': scores.mean(),
        'CV Std': scores.std()
    })
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

cv_df = pd.DataFrame(cv_results).sort_values('CV Mean ROC-AUC', ascending=False)
cv_df

### Hyperparameter Tuning (Top Model)

In [ ]:
# Select top model from CV
top_model_name = cv_df.iloc[0]['Model']
print(f'Tuning hyperparameters for: {top_model_name}\n')

if top_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10]
    }
    grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid,
                        cv=cv, scoring='roc_auc', n_jobs=-1)
    grid.fit(X_train, y_train)

elif top_model_name == 'XGBoost':
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 1.0]
    }
    grid = GridSearchCV(xgb.XGBClassifier(use_label_encoder=False,
                                          eval_metric='logloss',
                                          random_state=42),
                        param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    grid.fit(X_train, y_train)

elif top_model_name == 'LightGBM':
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7, -1],
        'learning_rate': [0.01, 0.1],
        'num_leaves': [31, 50]
    }
    grid = GridSearchCV(lgb.LGBMClassifier(random_state=42, verbose=-1),
                        param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    grid.fit(X_train, y_train)

else:
    # Fallback for Logistic Regression or SVM
    param_grid = {'C': [0.01, 0.1, 1, 10]}
    base_model = LogisticRegression(max_iter=1000, random_state=42) if top_model_name == 'Logistic Regression' else SVC(probability=True, random_state=42)
    grid = GridSearchCV(base_model, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    grid.fit(X_train_scaled if top_model_name in ['Logistic Regression', 'SVM'] else X_train, y_train)

print(f'Best Parameters: {grid.best_params_}')
print(f'Best CV ROC-AUC: {grid.best_score_:.4f}')
best_model = grid.best_estimator_

### Final Model Evaluation

In [ ]:
if top_model_name in ['Logistic Regression', 'SVM']:
    y_pred_final = best_model.predict(X_test_scaled)
    y_proba_final = best_model.predict_proba(X_test_scaled)[:, 1]
else:
    y_pred_final = best_model.predict(X_test)
    y_proba_final = best_model.predict_proba(X_test)[:, 1]

print('=== Final Model Performance ===\n')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_final):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_final):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_final):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba_final):.4f}')

print('\n=== Classification Report ===\n')
print(classification_report(y_test, y_pred_final, target_names=['No Diabetes', 'Diabetes']))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

### ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba_final)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'{top_model_name} (AUC = {roc_auc_score(y_test, y_proba_final):.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

### Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importance = np.abs(best_model.coef_[0])
else:
    importance = None

if importance is not None:
    feat_imp = pd.DataFrame({
        'Feature': X.columns,
        'Importance': importance
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='viridis')
    plt.title(f'Feature Importance - {top_model_name}')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importance not available for this model.')

### Save Best Model

In [ ]:
joblib.dump(best_model, '../best_model.pkl')
print(f'Best model ({top_model_name}) saved as best_model.pkl')

### Compare All Models - ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))

for name, model in trained_models.items():
    if name in ['Logistic Regression', 'SVM']:
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - All Models')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

### Model Explainability with SHAP

In [ ]:
try:
    import shap
    
    if top_model_name in ['Logistic Regression', 'SVM']:
        X_explain = X_test_scaled[:100]
    else:
        X_explain = X_test[:100]
    
    explainer = shap.TreeExplainer(best_model) if hasattr(best_model, 'feature_importances_') else shap.KernelExplainer(best_model.predict_proba, X_explain)
    shap_values = explainer.shap_values(X_explain)
    
    # Summary plot
    shap.summary_plot(shap_values if isinstance(shap_values, list) and len(shap_values) == 2 else shap_values,
                      X_explain, feature_names=X.columns.tolist())
    plt.show()
    
    # Bar plot of feature importance
    shap.summary_plot(shap_values if isinstance(shap_values, list) and len(shap_values) == 2 else shap_values,
                      X_explain, feature_names=X.columns.tolist(), plot_type='bar')
    plt.show()
    
    print('SHAP analysis completed successfully!')
except ImportError:
    print('SHAP not installed. Install with: pip install shap')

### Model Information Summary

In [ ]:
print('=== MODEL SUMMARY ===')
print(f'Best Model: {top_model_name}')
print(f'Best Parameters: {grid.best_params_}')
print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')
print(f'Features: {list(X.columns)}')
print(f'\nTest Performance:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_final):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred_final):.4f}')
print(f'  F1-Score:  {f1_score(y_test, y_pred_final):.4f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_proba_final):.4f}')

## Conclusion

- The best model has been selected based on cross-validated ROC-AUC.
- The model and scaler are saved for deployment in the Streamlit app.
- Glucose, BMI, and Age were consistently the most important predictors.
- SHAP analysis confirms feature contributions and provides interpretability.

### Next Steps
1. Run the Streamlit app: `streamlit run app/app.py`
2. Deploy to Hugging Face Spaces or Streamlit Cloud
3. Collect more data to improve model performance